# Experiment 1 — Final V3 Clean FA synthetic data

This notebook describes the exact 5,000-patient cohort used by the final Experiment 1 and both GI-without-NOW analyses. It reads data only: no generation, fitting, prediction, or writes to the dataset.

Synthetic patient IDs and ground truth are design metadata, not real clinical records. Ground truth and labels must not be included in the model-visible predictors.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data/temporal_benchmark_v3_clean_fa").is_dir())
DATA = ROOT / "data/temporal_benchmark_v3_clean_fa"
x = pd.read_pickle(DATA / "temporal_benchmark_5000.pkl")
fa = pd.read_pickle(DATA / "temporal_benchmark_fa_labels_5000.pkl")
gi = pd.read_pickle(DATA / "temporal_benchmark_gi_labels_5000.pkl")
assert list(x.columns) == ["patient_id", "time", "feature_name", "value"]
assert set(x.patient_id) == set(fa.patient_id) == set(gi.patient_id)
assert fa.patient_id.is_unique and gi.patient_id.is_unique


## Cohort, features, and class distributions


In [ ]:
display(pd.Series({"patients": x.patient_id.nunique(), "observations": len(x),
                   "features": x.feature_name.nunique()}, name="cohort"))
display(pd.DataFrame({"positive_count": [fa.label.sum(), gi.label.sum()],
                      "prevalence": [fa.label.mean(), gi.label.mean()]}, index=["FA", "GI"]))
display(x.groupby("feature_name").agg(observations=("patient_id", "size"),
                                     patients=("patient_id", "nunique")))


## Observation density and feature values


In [ ]:
anchor_rows = x.feature_name.str.endswith("Anchor")
events = x.loc[~anchor_rows].copy()
events["value"] = pd.to_numeric(events.value, errors="coerce")
display(events.groupby("patient_id").size().describe().to_frame("observations_per_patient"))
display(events.groupby("feature_name").value.describe())
fig, ax = plt.subplots(figsize=(7, 3))
events.groupby("patient_id").size().hist(bins=30, ax=ax)
ax.set(xlabel="Observations per synthetic patient", ylabel="Patients")
plt.show()


## Temporal structure relative to Birth and NOW


In [ ]:
birth = x.loc[x.feature_name.eq("Birth Date Anchor")].set_index("patient_id").value.map(pd.Timestamp)
now = x.loc[x.feature_name.eq("NOW Anchor")].set_index("patient_id").value.map(pd.Timestamp)
events["age_days"] = (pd.to_datetime(events.time) - events.patient_id.map(birth)).dt.total_seconds() / 86400
display(((now - birth).dt.total_seconds() / 86400).describe().to_frame("followup_days"))
fig, ax = plt.subplots(figsize=(9, 3))
for feature in ("FormulaYN", "BreastMilkYN", "GISymptomYN"):
    ax.hist(events.loc[events.feature_name.eq(feature), "age_days"], bins=30,
            alpha=0.4, label=feature)
ax.set(xlabel="Age in days", ylabel="Observation rows")
ax.legend()
plt.show()


## Transformation and acceptance audit

The final permutation preserves patient IDs, timestamps, labels, and the protected feeding/GI rows. It permutes selected distractor values independently of labels. “Clean FA” is a version name, not a guarantee that all screens pass.

The original predefined distractor acceptance screen **failed**: 12 sparse conditional summaries were flagged. The notebook reports the stored result without changing thresholds, data, or labels.


In [ ]:
validation = json.loads((DATA / "validation_summary.json").read_text())
display(pd.Series({key: validation[key] for key in (
    "patient_count", "distractor_acceptance_passed", "distractor_flagged_summaries",
    "late_feeding_flagged_summaries")}))
display(pd.read_csv(DATA / "before_after_audit.csv"))
